# v15 CNN系 5手法(学習あり, 5-fold)― v13生PFデータから各手法用の画像をその場で構築

v13 の生波形(`fwd_traj_*`/`bwd_traj_*`)+ center + target_rel + GR から、**手法ごとに異なる入力表現を
on-the-fly で構築**して学習する。線描画は「線が通った所を染めるだけ(生カウント・ぼかし無し)」。

手法:
1. **m1_unet**   : 2D U-Net(順密度/逆密度/GRコストの (3,H,T) 画像)→ ヒートマップ → soft-argmax
2. **m2_dilated**: 2D dilated all-conv(同画像, 解像度維持)→ ヒートマップ
3. **m3_bandch** : 1D-CNN。密度(H,T)を H=チャンネルとして MD方向1D畳み込み → オフセット回帰
4. **m4_summary**: 1D-TCN。各MDで順/逆の平均/分位点/差を F ch に要約 → 回帰
5. **m5_hybrid** : 2D画像エンコーダ(H方向平均集約)+ 1Dサマリ を結合 → 回帰

評価: GroupKFold(5) by well, OOF全行プールの overall RMSE(target_rel基準=TVT RMSE)。
学習: T方向ランダムクロップ(VRAM対策), 全長推論。重いので METHODS で対象を絞れる。

In [1]:
import time, copy, math
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

In [2]:
# ============================== CONFIG ==============================
PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
def _find(*c): return next((x for x in c if x.exists()), None)
RAW_DIR = _find(PROJECT/"v13"/"artifacts_v13"/"img", PROJECT/"artifacts_v13"/"img")
DATA_DIR = PROJECT / "rogii-wellbore-geology-prediction" / "train"
OUT_DIR = PROJECT/"v15"/"artifacts_v15"; OUT_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_DIR is not None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

METHODS = ["m1_unet","m2_dilated","m3_bandch","m4_summary","m5_hybrid"]   # 実行する手法(絞れる)
N_SPLITS = 5; SEED = 0
BAND = 32.0; BINW = 0.5; H = int(2*BAND/BINW)+1
OFF = (np.arange(H)-H//2)*BINW
CROP = 768; BATCH = 4; EPOCHS = 25; LR = 1e-3; WD = 1e-5; PATIENCE = 6
HID = 32
N_WELLS = None
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, " H:", H, " RAW:", RAW_DIR)
GRP = dict(HGR=3,TW=0,SIG_MIN=7.495879205722816,SIG_MAX=32.79346108797473,SIG_DEF=114.11439615641156,SIG_MULT=2.7482077023565075)

device: cuda  H: 129  RAW: C:\Users\kosaka256\Documents\rogii_claude\v13\artifacts_v13\img


In [3]:
# ---- 坑井メタ(center,target_rel,GR材料)をキャッシュ。波形はnpzから都度ロード ----
def _smooth(vals, fb, r):
    r=int(r); s=pd.Series(vals,dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r<=0 else s.rolling(2*r+1,center=True,min_periods=1).mean().to_numpy(np.float32)
def _grid(tw_tvt,tw_gr,step=0.2):
    a,b=float(tw_tvt.min()),float(tw_tvt.max()); g=np.arange(a,b+step,step)
    return np.interp(g,tw_tvt,tw_gr).astype(np.float64),float(a),float(step)
def gr_materials(wid, ev_index):
    hw=pd.read_csv(DATA_DIR/f"{wid}__horizontal_well.csv"); tw=pd.read_csv(DATA_DIR/f"{wid}__typewell.csv").sort_values("TVT")
    tw_tvt=tw["TVT"].to_numpy(float); tw_gr=tw["GR"].to_numpy(float)
    fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth(tw_gr,fb,GRP["TW"]).astype(np.float64); gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    hk=_smooth(hw["GR"].astype(float).to_numpy(),fb,GRP["HGR"]); kn=hw[hw["TVT_input"].notna()]
    if len(kn)<20: gs=GRP["SIG_DEF"]
    else:
        kp=hw.index.get_indexer(kn.index); rs=hk[kp]-np.interp(kn["TVT_input"].values,tw_tvt,tw_gr_pf)
        gs=float(np.nanstd(rs)); gs=GRP["SIG_DEF"] if (not np.isfinite(gs) or gs<=0) else gs
    gs=float(np.clip(gs*GRP["SIG_MULT"],GRP["SIG_MIN"],GRP["SIG_MAX"]))
    gr_full=hw["GR"].astype(float).interpolate(limit_direction="both").fillna(fb).to_numpy(float)
    gr_ev=_smooth(gr_full,fb,GRP["HGR"])[ev_index].astype(np.float32)
    return dict(gg=gg.astype(np.float32), gg_axis=(gmin+np.arange(len(gg))*gst).astype(np.float32), gs=np.float32(gs), gr_ev=gr_ev)

wells = sorted(p.stem for p in RAW_DIR.glob("*.npz"))
if N_WELLS: wells = wells[:N_WELLS]
CACHE = {}
print("メタ作成中...")
for i,w in enumerate(wells):
    d=np.load(RAW_DIR/f"{w}.npz")
    if "target_rel" not in d.files: continue
    ev=d["ev_index"].astype(np.int64)
    CACHE[w]=dict(center=d["center"].astype(np.float32), target_rel=d["target_rel"].astype(np.float32),
                  **gr_materials(w, ev))
    if (i+1)%150==0: print(f"  {i+1}/{len(wells)}")
wells=[w for w in wells if w in CACHE]
print("wells:", len(wells))

メタ作成中...
  150/773
  300/773
  450/773
  600/773
  750/773
wells: 773


In [4]:
# ---- on-the-fly 表現ビルダー(全て center相対) ----
def _load_traj(wid):
    d=np.load(RAW_DIR/f"{wid}.npz")
    fwd=np.concatenate([d[k] for k in d.files if k.startswith("fwd_traj_")] or [d["fwd_traj"]],0).astype(np.float32)
    bwd=np.concatenate([d[k] for k in d.files if k.startswith("bwd_traj_")] or [d["bwd_traj"]],0).astype(np.float32)
    return fwd, bwd      # (Sf,T),(Sb,T) 絶対TVT

def _paint(rel):        # rel:(S,Tc) → 密度(H,Tc) 線を染めるだけ(生カウント/S)
    S,Tc=rel.shape
    idx=np.clip(np.round(rel/BINW).astype(np.int64)+H//2,0,H-1)
    dens=np.zeros((H,Tc),np.float32); tcol=np.arange(Tc)
    for s in range(S): np.add.at(dens,(idx[s],tcol),np.float32(1.0/S))
    return dens

def _grcost(cache, ctr_sl):      # (H,Tc)
    tvt=ctr_sl[:,None]+OFF[None,:]; exp=np.interp(tvt.ravel(),cache["gg_axis"],cache["gg"]).reshape(len(ctr_sl),H)
    return np.clip(np.abs(cache["gr_ev_sl"][:,None]-exp)/(5*float(cache["gs"])+1e-6),0,1).T.astype(np.float32)

def build_img3(cache, fwd, bwd, sl):    # (3,H,Tc)
    c=cache["center"][sl]
    hf=_paint(fwd[:,sl]-c[None,:]); hb=_paint(bwd[:,sl]-c[None,:])
    cache["gr_ev_sl"]=cache["gr_ev"][sl]; hg=_grcost(cache,c)
    return np.stack([hf,hb,hg],0)

def build_bandch(cache, fwd, bwd, sl):  # (2H,Tc)
    c=cache["center"][sl]
    return np.concatenate([_paint(fwd[:,sl]-c[None,:]), _paint(bwd[:,sl]-c[None,:])],0)

def build_summary(cache, fwd, bwd, sl): # (F,Tc) PFサマリ(center相対ft)
    c=cache["center"][sl]; fr=fwd[:,sl]-c[None,:]; br=bwd[:,sl]-c[None,:]
    def q(a,p): return np.percentile(a,p,axis=0).astype(np.float32)
    feats=[fr.mean(0),fr.std(0),q(fr,10),q(fr,50),q(fr,90),
           br.mean(0),br.std(0),q(br,10),q(br,50),q(br,90),
           (fr.mean(0)-br.mean(0))]
    # GRコスト最小オフセット(セミブランス的)
    cache["gr_ev_sl"]=cache["gr_ev"][sl]; gc=_grcost(cache,c); feats.append(OFF[gc.argmin(0)].astype(np.float32))
    return np.stack(feats,0).astype(np.float32)

In [5]:
# ---- モデル ----
def conv_blk(ci,co,k=3,p=1,d=1):
    return nn.Sequential(nn.Conv2d(ci,co,k,padding=p,dilation=d), nn.BatchNorm2d(co), nn.GELU(),
                         nn.Conv2d(co,co,k,padding=p,dilation=d), nn.BatchNorm2d(co), nn.GELU())
class UNet2D(nn.Module):
    def __init__(s,C,h):
        super().__init__(); s.e1=conv_blk(C,h); s.e2=conv_blk(h,h*2); s.e3=conv_blk(h*2,h*4); s.b=conv_blk(h*4,h*4)
        s.d3=conv_blk(h*8,h*2); s.d2=conv_blk(h*4,h); s.d1=conv_blk(h*2,h); s.head=nn.Conv2d(h,1,1); s.pool=nn.MaxPool2d(2,ceil_mode=True)
    def up(s,x,r): return F.interpolate(x,size=r.shape[-2:],mode="nearest")
    def forward(s,x):
        e1=s.e1(x); e2=s.e2(s.pool(e1)); e3=s.e3(s.pool(e2)); b=s.b(s.pool(e3))
        d3=s.d3(torch.cat([s.up(b,e3),e3],1)); d2=s.d2(torch.cat([s.up(d3,e2),e2],1)); d1=s.d1(torch.cat([s.up(d2,e1),e1],1))
        return s.head(d1).squeeze(1)        # (B,H,T)
class Dilated2D(nn.Module):
    def __init__(s,C,h):
        super().__init__(); s.inp=nn.Conv2d(C,h,3,padding=1)
        s.blocks=nn.ModuleList([conv_blk(h,h,p=2**i,d=2**i) for i in range(4)]); s.head=nn.Conv2d(h,1,1)
    def forward(s,x):
        y=s.inp(x)
        for b in s.blocks: y=y+b(y)
        return s.head(y).squeeze(1)         # (B,H,T)
def conv1d_blk(ci,co,d):
    return nn.Sequential(nn.Conv1d(ci,co,3,padding=d,dilation=d), nn.GELU(), nn.Dropout(0.1),
                         nn.Conv1d(co,co,3,padding=d,dilation=d), nn.GELU())
class CNN1D(nn.Module):    # (B,Cin,T)->(B,T) 回帰
    def __init__(s,Cin,h,levels=7):
        super().__init__(); s.inp=nn.Conv1d(Cin,h,1); s.blocks=nn.ModuleList([conv1d_blk(h,h,2**i) for i in range(levels)]); s.head=nn.Conv1d(h,1,1)
    def forward(s,x):
        y=s.inp(x)
        for b in s.blocks: y=y+b(y)
        return s.head(y).squeeze(1)
class Hybrid(nn.Module):   # 2D画像(H平均集約)+ 1Dサマリ -> 回帰
    def __init__(s,Cimg,Fsum,h):
        super().__init__(); s.enc=nn.Sequential(nn.Conv2d(Cimg,h,3,padding=1),nn.GELU(),nn.Conv2d(h,h,3,padding=1),nn.GELU())
        s.tcn=CNN1D(h+Fsum,h,levels=6)
    def forward(s,img,summ):
        e=s.enc(img).mean(2)               # (B,h,T)  H方向平均
        return s.tcn(torch.cat([e,summ],1))

def soft_argmax(logits):                   # (B,H,T)->offset(ft)(B,T)
    bins=torch.arange(logits.shape[1],device=logits.device,dtype=logits.dtype).view(1,-1,1)
    p=torch.softmax(logits,1); return ((p*bins).sum(1)-logits.shape[1]//2)*BINW

In [6]:
# ---- 手法レジストリ(入力ビルダー / モデル生成 / 出力種別) ----
def make_model(method):
    if method=="m1_unet":    return UNet2D(3,HID), "img3", "heat"
    if method=="m2_dilated": return Dilated2D(3,HID), "img3", "heat"
    if method=="m3_bandch":  return CNN1D(2*H,HID), "bandch", "reg"
    if method=="m4_summary": return CNN1D(12,HID), "summary", "reg"
    if method=="m5_hybrid":  return None, "hybrid", "reg"   # 特殊(2入力)
    raise ValueError(method)

def build_input(kind, cache, fwd, bwd, sl):
    if kind=="img3":    return build_img3(cache,fwd,bwd,sl)
    if kind=="bandch":  return build_bandch(cache,fwd,bwd,sl)
    if kind=="summary": return build_summary(cache,fwd,bwd,sl)
    if kind=="hybrid":  return (build_img3(cache,fwd,bwd,sl), build_summary(cache,fwd,bwd,sl))

class WellDS(Dataset):
    def __init__(s,well_list,kind,out): s.w=list(well_list); s.kind=kind; s.out=out
    def __len__(s): return len(s.w)
    def __getitem__(s,i):
        wid=s.w[i]; ca=CACHE[wid]; fwd,bwd=_load_traj(wid); T=len(ca["center"])
        if T>=CROP: a=np.random.randint(0,T-CROP+1); sl=slice(a,a+CROP); L=CROP
        else: sl=slice(0,T); L=T
        x=build_input(s.kind,ca,fwd,bwd,sl)
        tr=ca["target_rel"][sl]; msk=(np.abs(tr)<=BAND).astype(np.float32)
        # pad to CROP
        def padT(a2,Lc):
            if a2.ndim==1: o=np.zeros(Lc,a2.dtype); o[:len(a2)]=a2; return o
            o=np.zeros(a2.shape[:-1]+(Lc,),a2.dtype); o[...,:a2.shape[-1]]=a2; return o
        if s.kind=="hybrid":
            img,su=x; img=padT(img,CROP); su=padT(su,CROP)
            xb=(torch.from_numpy(img),torch.from_numpy(su))
        else:
            xb=torch.from_numpy(padT(x,CROP))
        tr=padT(tr,CROP); msk=padT(msk,CROP)
        if s.out=="heat":
            tb=np.clip(np.round(tr/BINW).astype(np.int64)+H//2,0,H-1)
            return xb, torch.from_numpy(tb), torch.from_numpy(tr), torch.from_numpy(msk)
        return xb, torch.from_numpy(tr), torch.from_numpy(tr), torch.from_numpy(msk)

In [7]:
def forward_model(method, model, xb, device):
    if method=="m5_hybrid":
        img,su=xb; return model(img.to(device), su.to(device))
    return model(xb.to(device))

def predict_off(method, model, wid, kind, out, device):
    ca=CACHE[wid]; fwd,bwd=_load_traj(wid); sl=slice(0,len(ca["center"]))
    x=build_input(kind,ca,fwd,bwd,sl)
    model.eval()
    with torch.no_grad():
        if kind=="hybrid":
            img,su=x; o=model(torch.from_numpy(img).unsqueeze(0).to(device), torch.from_numpy(su).unsqueeze(0).to(device))
        else:
            o=model(torch.from_numpy(x).unsqueeze(0).to(device))
        o=o.squeeze(0)
        off=soft_argmax(o.unsqueeze(0)).squeeze(0) if out=="heat" else o
    return off.float().cpu().numpy()

def overall_rmse(model, method, wl, kind, out, device):
    se=0.0; n=0
    for w in wl:
        pr=predict_off(method,model,w,kind,out,device); tr=CACHE[w]["target_rel"]
        se+=float(np.sum((pr-tr)**2)); n+=len(tr)
    return math.sqrt(se/max(n,1))

def run_fold(method, kind, out, tr_w, va_w, device):
    model,_,_ = (make_model(method)[0], None, None) if method!="m5_hybrid" else (Hybrid(3,12,HID),None,None)
    if model is None: model=Hybrid(3,12,HID)
    model=model.to(device)
    ds=WellDS(tr_w,kind,out); loader=DataLoader(ds,batch_size=BATCH,shuffle=True,num_workers=0,drop_last=True)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    best=1e18; best_state=None; pat=0
    for ep in range(EPOCHS):
        model.train()
        for batch in loader:
            xb,t1,tr,msk=batch; t1=t1.to(device); tr=tr.to(device); msk=msk.to(device)
            o=forward_model(method,model,xb,device)
            if out=="heat":
                ce=F.cross_entropy(o,t1,reduction="none"); ce=(ce*msk).sum()/msk.sum().clamp_min(1)
                off=soft_argmax(o); mse=(((off-tr)**2)*msk).sum()/msk.sum().clamp_min(1)
                loss=ce+0.05*mse
            else:
                loss=((F.smooth_l1_loss(o,tr,reduction="none"))*msk).sum()/msk.sum().clamp_min(1)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        vr=overall_rmse(model,method,va_w,kind,out,device)
        if vr<best-1e-4: best=vr; best_state=copy.deepcopy(model.state_dict()); pat=0
        else: pat+=1
        print(f"    ep{ep:2d} val_overall={vr:.4f} (best={best:.4f})")
        if pat>=PATIENCE: break
    model.load_state_dict(best_state)
    return {w:predict_off(method,model,w,kind,out,device) for w in va_w}, best

In [12]:
# ---- 5-fold × 各手法 ----
wells_arr=np.array(wells); gkf=GroupKFold(n_splits=N_SPLITS)
results={}
for method in METHODS[4:]:
    _,kind,out = make_model(method)
    print(f"\n########## {method} (kind={kind}, out={out}) ##########")
    oof={}; folds=[]; t0=time.perf_counter()
    for k,(tr,va) in enumerate(gkf.split(wells_arr,groups=wells_arr)):
        print(f"  -- fold {k} --")
        preds,best=run_fold(method,kind,out,wells_arr[tr],wells_arr[va],DEVICE)
        oof.update(preds); folds.append(best)
    se=0.0;n=0
    for w in wells: se+=float(np.sum((oof[w]-CACHE[w]["target_rel"])**2)); n+=len(CACHE[w]["target_rel"])
    ov=math.sqrt(se/n)
    np.savez_compressed(OUT_DIR/f"oof_{method}.npz", **{w:oof[w].astype(np.float32) for w in wells})
    results[method]=dict(overall=ov, folds=folds, time=time.perf_counter()-t0)
    print(f"  >>> {method}: overall RMSE={ov:.4f}  ({results[method]['time']/60:.1f}min)")


########## m5_hybrid (kind=hybrid, out=reg) ##########
  -- fold 0 --
    ep 0 val_overall=8.9819 (best=8.9819)
    ep 1 val_overall=9.0128 (best=8.9819)
    ep 2 val_overall=9.1081 (best=8.9819)
    ep 3 val_overall=9.0634 (best=8.9819)
    ep 4 val_overall=8.7791 (best=8.7791)
    ep 5 val_overall=9.1950 (best=8.7791)
    ep 6 val_overall=8.7518 (best=8.7518)
    ep 7 val_overall=8.9758 (best=8.7518)
    ep 8 val_overall=9.2954 (best=8.7518)
    ep 9 val_overall=8.8901 (best=8.7518)
    ep10 val_overall=8.9677 (best=8.7518)
    ep11 val_overall=8.8202 (best=8.7518)
    ep12 val_overall=8.8991 (best=8.7518)
  -- fold 1 --
    ep 0 val_overall=8.8071 (best=8.8071)
    ep 1 val_overall=8.8242 (best=8.8071)
    ep 2 val_overall=9.3612 (best=8.8071)
    ep 3 val_overall=9.1475 (best=8.8071)
    ep 4 val_overall=8.8519 (best=8.8071)
    ep 5 val_overall=8.9133 (best=8.8071)
    ep 6 val_overall=9.0544 (best=8.8071)
  -- fold 2 --
    ep 0 val_overall=8.0763 (best=8.0763)
    ep 1 val_over

In [ ]:
rows=[(m,r["overall"],np.mean(r["folds"])) for m,r in results.items()]
df=pd.DataFrame(rows,columns=["method","overall_rmse","mean_fold"]).sort_values("overall_rmse").reset_index(drop=True)
print("参考: v10(LGBM,2)=7.5537 / v12(TCN)=7.8710 / v13 U-Net(pf_1)=8.5829 / pf_ancc=9.1877")
df